# Environment and packaged inputs

Run this on the lab computer before the model notebooks. This package contains
the original 45 validation images, original trained best-accuracy checkpoints,
and archived predictions. It does not contain new training runs or explanations.
Read [START_HERE.md](START_HERE.md) for installation, particularly VMamba.


In [1]:
from pathlib import Path
import sys
import numpy as np
import torch

# Usually leave this as None. Set a path only if Jupyter starts elsewhere.
PACKAGE_ROOT = None
if PACKAGE_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p/'package_integrity.json').is_file()
                 and (p/'xai_code').is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError('Start Jupyter inside the extracted package, or set PACKAGE_ROOT.')
else:
    ROOT = Path(PACKAGE_ROOT).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from xai_code.models import build_model, Predictor
from xai_code.runtime import prepare, audit_predictions, verify_package, environment, samples
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Package:', ROOT)
print('Device:', DEVICE)


Package: /home/murat/Desktop/original_bandwidth_explainability_lab
Device: cuda


In [2]:
import json
verify_package(ROOT)
print(json.dumps(environment(), indent=2))
assert len(samples(ROOT)) == 45
print('45 validation images; classes 0=Low bandwidth, 1=High bandwidth')
if DEVICE.type == 'cuda':
    value = torch.randn(32, 32, device=DEVICE)
    assert torch.isfinite(value @ value.T).all()
    print('Basic CUDA operation passed:', torch.cuda.get_device_name(0))
else:
    print('CUDA is unavailable. Resolve this before VMamba LIME; other models support CPU.')


Integrity OK: 84 fixed files; notebook existence checked separately
{
  "python": "3.12.15",
  "system": "Linux",
  "versions": {
    "torch": "2.12.0+cu130",
    "torchvision": "0.27.0+cu130",
    "timm": "1.0.20",
    "lime": "0.2.0.1",
    "scikit-image": "0.25.2",
    "numpy": "2.5.2",
    "scikit-learn": "1.9.1",
    "Pillow": "12.3.0"
  },
  "cuda_runtime": "13.0",
  "gpu": "NVIDIA GeForce RTX 5050 Laptop GPU"
}
45 validation images; classes 0=Low bandwidth, 1=High bandwidth
Basic CUDA operation passed: NVIDIA GeForce RTX 5050 Laptop GPU


## Optional VMamba scan check

After installing the CUDA extension, set `CHECK_VMAMBA_KERNEL=True` and run
this cell. It checks a small synthetic forward against the reference scan;
it does not train a model or explain a validation image.


In [3]:
CHECK_VMAMBA_KERNEL = True
if CHECK_VMAMBA_KERNEL:
    from xai_code import vmamba_official as official
    if DEVICE.type != 'cuda' or not official.WITH_SELECTIVESCAN_MAMBA:
        raise RuntimeError('CUDA / selective_scan_cuda missing; see START_HERE.md')
    torch.manual_seed(42)
    values = [torch.randn((1, 8, 16), device=DEVICE)*0.1,
              torch.randn((1, 8, 16), device=DEVICE)*0.1,
              -torch.rand((8, 1), device=DEVICE),
              torch.randn((1, 4, 1, 16), device=DEVICE)*0.1,
              torch.randn((1, 4, 1, 16), device=DEVICE)*0.1,
              torch.randn(8, device=DEVICE)*0.1,
              torch.randn(8, device=DEVICE)*0.1]
    with torch.inference_mode():
        reference = official.selective_scan_fn(*values, delta_softplus=True, oflex=True, backend='torch')
        actual = official.selective_scan_fn(*values, delta_softplus=True, oflex=True, backend='mamba')
    torch.testing.assert_close(actual.float(), reference.float(), rtol=1e-4, atol=1e-4)
    print('VMamba CUDA scan matches the float32 reference on synthetic inputs.')
else:
    print('Optional VMamba check skipped; enable it before running VMamba LIME.')


/home/murat/Desktop/original_bandwidth_explainability_lab/xai_code/vmamba_official.py:853: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/home/murat/Desktop/original_bandwidth_explainability_lab/xai_code/vmamba_official.py:868: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


VMamba CUDA scan matches the float32 reference on synthetic inputs.


/home/murat/Desktop/training_models/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Next, open a model notebook and use **Run All**. Start with CNN LIME.
Each model first audits all 45 predictions against the saved results. Keep the
package folder together when transferring it; absolute paths from the old
computer are not needed.